In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pickle

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [ ]:
df = pd.read_csv("/content/Combined_dataset_model.csv")
df = pd.get_dummies(df, columns=["biome"], dtype=int)
df = df.drop(columns=['Unnamed: 0'])
print(df.columns)


Index(['county', 'land_area', 'tc_goal', 'treecanopy', 'tc_gap', 'priority_i',
       'pctpocnorm', 'pctpovnorm', 'unemplnorm', 'dep_perc', 'depratnorm',
       'health_nor', 'temp_norm', 'tes', 'tesctyscor', 'rank', 'rankgrpsz',
       'Mean_Temp', 'Median_Temp', 'STD_Temp', 'Min_Temp', 'Max_Temp',
       'Mean_Rain', 'Median_Rain', 'STD_Rain', 'Min_Rain', 'Max_Rain',
       'biome_Desert', 'biome_Forest', 'biome_Grassland'],
      dtype='object')


In [ ]:
features = ['land_area', 'treecanopy', 'tc_gap',
       'priority_i', 'pctpocnorm', 'pctpovnorm', 'unemplnorm', 'dep_perc',
       'depratnorm', 'health_nor', 'tes', 'tesctyscor', 'rank',
       'rankgrpsz', 'Mean_Temp', 'Median_Temp', 'STD_Temp', 'Min_Temp',
       'Max_Temp', 'Mean_Rain', 'Median_Rain', 'STD_Rain', 'Min_Rain',
       'Max_Rain', 'biome_Desert', 'biome_Forest', 'biome_Grassland']
target = ['temp_norm']
X_df = df[features]
y_df = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size = 0.2, random_state = 42)

In [ ]:
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
}

grid_search = GridSearchCV(
    XGBRegressor(random_state = 42, objective='reg:squarederror'),
    param_grid,
    cv = 5,
    scoring = 'r2'
)

grid_search.fit(X_train, y_train)

print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))
print("Test acc    :", round(grid_search.best_estimator_.score(X_test, y_test), 4))

In [ ]:
best_model = grid_search.best_params_
best_model

In [ ]:
model = XGBRegressor(learning_rate = 0.2, max_depth = 4, n_estimators = 300)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)


#model = pickle.load(open('xgb_model_pickle', 'rb'))
print("R2 score:", r2_score(y_test, model_pred))
print(np.var(model_pred))



R2 score: 0.6676608324050903
0.022109233


In [ ]:
solution_costs = pd.read_csv("/content/Green Intervention Budgets - Sheet1.csv")

In [ ]:
# Define what each solution actually changes, and by how much
solution_effects = {
    "urban_forest": {
        "treecanopy": 50, # adds 300 points of canopy
        "Mean_Temp": -10,
        "Cost": int(solution_costs[solution_costs['intervention'] == "Urban Forest"]['sq_foot_cost_dollars'].iloc[0] * solution_costs[solution_costs['intervention'] == "Urban Forest"]["avg_sq_feet"].iloc[0])

    },

    "park": {
        "treecanopy": 50,
        "Mean_Temp": -30,  # Changed from 'Mean Temp' to 'Mean_Temp'
        "Cost": int(solution_costs[solution_costs['intervention'] == "Park"]['sq_foot_cost_dollars'].iloc[0] * solution_costs[solution_costs['intervention'] == "Park"]["avg_sq_feet"].iloc[0])
    }
}

solution_effects = pd.DataFrame(solution_effects)
solution_effects = solution_effects.T
solution_effects.head()

NameError: name 'solution_costs' is not defined

In [ ]:
def impact_calc(model, county, solutions, feature_cols):
  impact_df = []
  county_df = df[df["county"] == county]
  for row in county_df:
    prediction = model.predict(row)
    for solution in solution_effects[solution].items():
      for key, value in solution_effects[solution].items():
        county_df[key] = county_df[key] + value

    update_prediction = model.predict(row)
    pct_hbi_chg = update_prediction/prediction
    impact_df.concat(county, solution, pct_hbi_chg)
  return impact_df


test = impact_calc(model, "Austin County", solution_effects, )

In [ ]:
def impact_calc(model, county, solutions_dict, feature_cols):
  impact_results = [] # To store results
  county_df_filtered = df[df["county"] == county]

  for index, row_data in county_df_filtered.iterrows():
    # Get prediction
    original_features = row_data[feature_cols]
    prediction_original = model.predict(pd.DataFrame([original_features]))[0]

    for solution_name, effects in solutions_dict.iterrows(): # Iterate through solutions and their effects
      modified_features = original_features.copy() # Create a copy to modify
      for key, value in effects.items(): # Apply effects
        if key in modified_features.index:
          modified_features[key] += value
        #else:
          #print("Warning: Feature", key, "from solution", solution_name, "not found in model features. Skipping update for this feature.")

      # Predict with modified features
      prediction_modified = model.predict(pd.DataFrame([modified_features]))[0]

      # Calculate change
      pct_hbi_chg = (prediction_modified - prediction_original) / prediction_original if prediction_original != 0 else 0

      impact_results.append({
          "county": county,
          "row_index": index,
          "solution": solution_name,
          "original_health_nor_pred": prediction_original,
          "modified_health_nor_pred": prediction_modified,
          "pct_hbi_change": pct_hbi_chg
      })
  return impact_results


result = impact_calc(model, "Austin County", solution_effects, features)
pd.DataFrame(result)
result

[{'county': 'Austin County',
  'row_index': 114,
  'solution': 'urban_forest',
  'original_health_nor_pred': np.float32(0.45750242),
  'modified_health_nor_pred': np.float32(0.6108859),
  'pct_hbi_change': np.float32(0.3352627)},
 {'county': 'Austin County',
  'row_index': 114,
  'solution': 'park',
  'original_health_nor_pred': np.float32(0.45750242),
  'modified_health_nor_pred': np.float32(0.6108859),
  'pct_hbi_change': np.float32(0.3352627)},
 {'county': 'Austin County',
  'row_index': 115,
  'solution': 'urban_forest',
  'original_health_nor_pred': np.float32(0.39504886),
  'modified_health_nor_pred': np.float32(0.48056418),
  'pct_hbi_change': np.float32(0.21646771)},
 {'county': 'Austin County',
  'row_index': 115,
  'solution': 'park',
  'original_health_nor_pred': np.float32(0.39504886),
  'modified_health_nor_pred': np.float32(0.48056418),
  'pct_hbi_change': np.float32(0.21646771)},
 {'county': 'Austin County',
  'row_index': 116,
  'solution': 'urban_forest',
  'original_h

In [ ]:
costs = [c for c in solution_effects['Cost']]
costs

[500000, 750000]

In [ ]:
from ortools.sat.python import cp_model

# Precomputed from XGBoost model.predict() for each candidate
impacts = result   # predicted health impact
costs = [c for c in solution_effects['Cost']]
total_cost = 0
budget  = 1200000

model_cp = cp_model.CpModel()

gstreet = model_cp.new_int_var(0, 5, "gstreet")
gparklot = model_cp.new_int_var(0, 5, "gparklot")
urbforest = model_cp.new_int_var(0, 5, "urbforest")
groof = model.new_int_var(0, 5, "groof")
gbelt = model.new_int_var(0, 5, "gbelt")
park = model.new_int_var(0, 5, "park")
garden = model.new_int_var(0, 5, "garden")

model_cp.Add(gstreet * 60000 + gparklot * 300000 + urbforest * 500000 <= budget)

# Define the objective function for maximization
model_cp.maximize(gstreet * 0.1 + gparklot * 0.06 + urbforest * 0.2)

solver = cp_model.CpSolver()
status = solver.Solve(model_cp) # Corrected to use model_cp

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(f"Optimal solutions found!")
    print(f"  Green Street interventions: {solver.Value(gstreet)}")
    print(f"  Green Parking Lot interventions: {solver.Value(gparklot)}")
    print(f"  Urban Forest interventions: {solver.Value(urbforest)}")
    print(f"  Total impact: {solver.ObjectiveValue()}")
    print(f"  Total cost: {solver.Value(gstreet) * 60000 + solver.Value(gparklot) * 300000 + solver.Value(urbforest) * 500000}")
elif status == cp_model.INFEASIBLE:
    print("No solution found that satisfies the constraints.")
else:
    print("Solver could not find an optimal or feasible solution.")

Optimal solutions found!
  Green Street interventions: 5
  Green Parking Lot interventions: 1
  Urban Forest interventions: 1
  Total impact: 0.76
  Total cost: 1100000
